# 🔩 Project 6.3 — CNC Toolpath Expression Tree
**DSA for Mechatronics · Week 06 — Trees & BSTs**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
CNC parametric programming uses **expression trees** to evaluate coordinate formulas.
For example, the feed rate formula `(A + B) * (C - D)` is stored as a tree
where operators are internal nodes and operands are leaves.

**Postorder traversal** evaluates the expression (children before parent).
**Inorder traversal** reconstructs the human-readable infix formula.

Your tree represents a unique machining formula seeded from your student ID.


## Step 1 — Build your expression tree

In [ ]:
class ENode:
    """A node in an expression tree."""
    def __init__(self, val, left=None, right=None):
        self.val   = val     # operator (+,-,*,/) or numeric operand
        self.left  = left
        self.right = right
    def is_leaf(self): return self.left is None and self.right is None
    def __repr__(self): return f"ENode({self.val})"

# ── Build a random expression tree ────────────────────────────────
OPS    = ['+', '-', '*', '/']
DEPTHS = random.randint(3, 5)   # tree depth

def rand_operand():
    return round(random.uniform(1.0, 50.0), 1)

def build_expr_tree(depth):
    """Recursively build a random expression tree."""
    if depth == 0 or (depth < DEPTHS and random.random() < 0.25):
        return ENode(rand_operand())
    op    = random.choice(OPS)
    left  = build_expr_tree(depth - 1)
    right = build_expr_tree(depth - 1)
    return ENode(op, left, right)

random.seed(_seed + 7777)   # sub-seed for expression
expr_root = build_expr_tree(DEPTHS)
random.seed(_seed)           # restore main seed

def count_nodes(node):
    if not node: return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def count_leaves(node):
    if not node: return 0
    if node.is_leaf(): return 1
    return count_leaves(node.left) + count_leaves(node.right)

def tree_height(node):
    if not node: return 0
    return 1 + max(tree_height(node.left), tree_height(node.right))

n_nodes  = count_nodes(expr_root)
n_leaves = count_leaves(expr_root)
h        = tree_height(expr_root)

print(f"Expression tree parameters:")
print(f"  Target depth   : {DEPTHS}")
print(f"  Actual nodes   : {n_nodes}")
print(f"  Leaf operands  : {n_leaves}")
print(f"  Height         : {h}")
print(f"  Root operator  : '{expr_root.val}'")


## Step 2 — Postorder evaluation and inorder infix reconstruction

In [ ]:
def evaluate(node):
    """
    Evaluate the expression using POSTORDER traversal:
    evaluate left subtree, evaluate right subtree, then apply operator.
    Returns float result, or raises ZeroDivisionError.
    """
    if node.is_leaf():
        return float(node.val)
    left_val  = evaluate(node.left)
    right_val = evaluate(node.right)
    if node.val == '+': return left_val + right_val
    if node.val == '-': return left_val - right_val
    if node.val == '*': return left_val * right_val
    if node.val == '/':
        if abs(right_val) < 1e-9:
            return float('inf')   # avoid crash, flag as invalid
        return left_val / right_val
    raise ValueError(f"Unknown operator: {node.val}")

def infix(node):
    """
    Reconstruct infix expression using INORDER traversal.
    Adds parentheses around every sub-expression for clarity.
    """
    if node.is_leaf():
        return str(node.val)
    return f"({infix(node.left)} {node.val} {infix(node.right)})"

def postfix(node):
    """Postorder traversal → RPN (Reverse Polish Notation)."""
    if node.is_leaf():
        return str(node.val)
    return f"{postfix(node.left)} {postfix(node.right)} {node.val}"

result    = evaluate(expr_root)
infix_str = infix(expr_root)
rpn_str   = postfix(expr_root)

print("Expression evaluation:")
print(f"  Infix (inorder)  : {infix_str}")
print(f"  RPN (postorder)  : {rpn_str}")
print(f"  Result           : {result:.4f}")
print()
if abs(result) > 1e6 or result == float('inf'):
    print("  ⚠  Result is very large or undefined (division by near-zero)")
else:
    print(f"  ✅ Valid machining parameter: {result:.4f}")


## Step 3 — Diameter: longest path in the expression tree

In [ ]:
def tree_diameter(node):
    """
    Diameter = number of edges on the longest path between any two nodes.
    At each node, the longest path through it equals left_height + right_height.
    Computed in a single O(n) pass.
    """
    best = [0]
    def height(n):
        if not n: return 0
        lh = height(n.left)
        rh = height(n.right)
        best[0] = max(best[0], lh + rh)
        return 1 + max(lh, rh)
    height(node)
    return best[0]

def collect_leaves(node, path=None, leaves=None):
    """Collect all leaf nodes with their root-to-leaf paths."""
    if path   is None: path   = []
    if leaves is None: leaves = []
    path = path + [node.val]
    if node.is_leaf():
        leaves.append((node.val, path))
        return leaves
    if node.left:  collect_leaves(node.left,  path, leaves)
    if node.right: collect_leaves(node.right, path, leaves)
    return leaves

diameter = tree_diameter(expr_root)
leaves   = collect_leaves(expr_root)

print(f"Tree structural analysis:")
print(f"  Diameter (longest path) : {diameter} edges")
print(f"  Number of leaf operands : {len(leaves)}")
print()
print("  Leaf operands and their root-to-leaf paths:")
print(f"  {'Operand':>10}  Path")
print(f"  {'─'*10}  {'─'*40}")
for val, path in leaves[:8]:
    print(f"  {str(val):>10}  {' → '.join(str(v) for v in path)}")
if len(leaves) > 8:
    print(f"  ... and {len(leaves)-8} more leaves")


## Step 4 — Serialize and deserialize the expression tree

In [ ]:
def serialize(node):
    """
    Serialize tree to a string using preorder traversal with null markers.
    Format: val,val,#,#,val,...   where '#' marks a None child.
    """
    if not node: return "#"
    return f"{node.val},{serialize(node.left)},{serialize(node.right)}"

def deserialize(data):
    """Reconstruct the tree from the serialized string."""
    tokens = iter(data.split(","))
    def _build():
        val = next(tokens)
        if val == "#": return None
        try:    val = float(val)
        except: pass   # keep as operator string
        node = ENode(val)
        node.left  = _build()
        node.right = _build()
        return node
    return _build()

serial_str   = serialize(expr_root)
rebuilt_root = deserialize(serial_str)
rebuilt_result = evaluate(rebuilt_root)
serial_len   = len(serial_str)

# Verify round-trip
ok = abs(result - rebuilt_result) < 1e-6 or (result == float('inf') and rebuilt_result == float('inf'))

print(f"Serialization round-trip:")
print(f"  Serialized length : {serial_len} characters")
print(f"  Serial preview    : {serial_str[:80]}{'...' if serial_len > 80 else ''}")
print()
print(f"  Original result   : {result:.6f}")
print(f"  Rebuilt result    : {rebuilt_result:.6f}")
print(f"  Round-trip match  : {'✅ YES' if ok else '❌ NO'}")
print()
print(f"  Rebuilt infix     : {infix(rebuilt_root)[:80]}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " CNC EXPRESSION TREE — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  TREE STRUCTURE" + " "*(W-16) + "║")
print(f"║  {'Nodes':<28}: {n_nodes:<26}║")
print(f"║  {'Leaf operands':<28}: {n_leaves:<26}║")
print(f"║  {'Height':<28}: {h:<26}║")
print(f"║  {'Diameter':<28}: {diameter} edges{'':<20}║")
print("╠" + "═"*W + "╣")
print("║  EVALUATION" + " "*(W-12) + "║")
infix_preview = infix_str[:48] + "…" if len(infix_str) > 48 else infix_str
print(f"║  {'Infix expression':<28}: {infix_preview:<26}║")
print(f"║  {'Result':<28}: {result:.4f}{'':<21}║")
print(f"║  {'Valid result':<28}: {'YES' if result != float('inf') else 'NO (div by zero)':<26}║")
print("╠" + "═"*W + "╣")
print("║  SERIALIZATION" + " "*(W-15) + "║")
print(f"║  {'Serialized length':<28}: {serial_len} chars{'':<21}║")
print(f"║  {'Round-trip verified':<28}: {'YES ✅' if ok else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which tree concept did you find most important, and why?**
Pick one technique from the notebook (traversal, BST property, recursion pattern…) and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
